# EDA

In [42]:
import json
import pandas as pd
import re

with open("../data/raw_ads_2511.json", encoding="utf-8") as f:
    df = pd.DataFrame(json.load(f))

In [44]:
with open("../data/skills_2511.json", encoding="utf-8") as f:
    tax = pd.DataFrame(json.load(f))

tax.head()

,skill_id,skill,headlines
0,13ad_dCC_Tn6,"OOS, styr- och utvecklingsmodell",[Styr- och utvecklingsmodeller]
1,16ZN_3Zp_bYV,"Odoo, molnplattform",[Applikationsplattformar]
2,19PK_uxU_Wzy,"Bower, pakethanterare",[Programmerings- och systemutvecklingsverktyg]
3,1CmG_8hR_igc,Bluetooth-teknik,[Kommunikationsprotokoll]
4,1QYQ_Xka_TQ5,"Microsoft Autopilot, systemutvecklingsverktyg",[Programmerings- och systemutvecklingsverktyg]


## Jobbannonser EDA

### Datasetets grundläggande struktur

In [45]:
# Snabb överblick
print(f"Datasetet innehåller {df.shape[0]} rader (annonser) och {df.shape[1]} kolumner.\n")
print("---------------------------")
print(df.info())
print("---------------------------")

print("\n Dataset kolumner utdrag:")
print(df.head())
print("---------------------------")

print("\nFeatures i datasetet:\n", df.columns)
print("---------------------------")

Datasetet innehåller 11345 rader (annonser) och 10 kolumner.

---------------------------
<class 'pandas.DataFrame'>
RangeIndex: 11345 entries, 0 to 11344
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   id            11345 non-null  str  
 1   version_id    11345 non-null  str  
 2   title         11345 non-null  str  
 3   employer      11345 non-null  str  
 4   municipality  9910 non-null   str  
 5   region        9910 non-null   str  
 6   occupation    11345 non-null  str  
 7   published     11345 non-null  str  
 8   deadline      11345 non-null  str  
 9   description   11345 non-null  str  
dtypes: str(10)
memory usage: 886.5 KB
None
---------------------------

 Dataset kolumner utdrag:
         id                                version_id  \
0  29427644  0a8a4a1038a8146a044682a6c77b69076a0c689d   
1  29403435  646b1b1099909ef1d5cb13571309ea9da0986b2e   
2  29403435  969cc4bc28a2090b1bb17611052aaddc80d1b

### Datakvalitet

In [57]:
# Saknade värden + tomma rader
missing = (
    df.isna() |
    df.astype(str).apply(lambda col: col.str.strip().eq(""))
).sum()

print(missing)

print("---------------------------")

missing_percentage = (
    missing / len(df) * 100
).sort_values(ascending=False)

print(missing_percentage)
print("---------------------------")


id                       0
version_id               0
title                    0
employer                 0
municipality          1435
region                1435
occupation               0
published                0
deadline                 0
description             14
year                     0
month                    0
year_month               0
description_length       0
word_count               0
dtype: int64
---------------------------
municipality          12.648744
region                12.648744
description            0.123402
title                  0.000000
version_id             0.000000
employer               0.000000
id                     0.000000
occupation             0.000000
published              0.000000
deadline               0.000000
year                   0.000000
month                  0.000000
year_month             0.000000
description_length     0.000000
word_count             0.000000
dtype: float64
---------------------------


In [60]:
#Dubletter
print("Unika annons-ID:", df["id"].nunique())
print("Totalt antal annonser:", len(df))
print("Antal duplicerade rader:", df.duplicated().sum())


# dubletter baserade på tid + arbetsgivare + beskrivning
duplicate_ads = df.duplicated(
    subset=["title", "employer", "description"]
).sum()

print("Potentiella duplicerade annonser:", duplicate_ads) 
print("---------------------------")

duplicate_texts = df[df["description"].duplicated(keep=False)]
duplicate_texts[["id","title","employer","published","description"]].head(20)
print("Potentiella duplicerade annonser:", duplicate_texts.sum)
print("---------------------------")

Unika annons-ID: 6778
Totalt antal annonser: 11345
Antal duplicerade rader: 4310
Potentiella duplicerade annonser: 5259
---------------------------
Potentiella duplicerade annonser: <bound method DataFrame.sum of              id                                version_id  \
1      29403435  646b1b1099909ef1d5cb13571309ea9da0986b2e   
2      29403435  969cc4bc28a2090b1bb17611052aaddc80d1bbfe   
3      29403435  9e26031d28b6d5255204909d767f11d367f90a48   
4      29403435  5e5363fbd7ed8ab6f5a52b7e1bd40f8a0f1b92de   
5      29427256  415760cd3cbdb012286f246b00a831a96649754f   
...         ...                                       ...   
11340  31423626  e1b65939b6213405bfdba3a8982ee505c8512406   
11341  31423252  0dc829603124ffcfee948f826716becbe8035d7f   
11342  31422467  8a7488519535bf268ae07169f84d188b901ed9bf   
11343  31421464  599c8df43ae9927b3e99df411c08bbd01d154a45   
11344  31420874  fa9900b9009727640393bb36fbc5263e60ca1f42   

                                                   tit

#### Fördelning av jobbannonser

In [48]:
#Titlar
unique_titles = df["title"].nunique()

print(f"Antal annonser: {len(df)}")
print(f"Antal unika jobbtitlar: {unique_titles}")
print(f"Genomsnittligt antal annonser per titel: {len(df) / unique_titles:.2f}")
print("---------------------------")

print(df["title"].value_counts().head(20))
print("---------------------------")

Antal annonser: 11345
Antal unika jobbtitlar: 4645
Genomsnittligt antal annonser per titel: 2.44
---------------------------
title
Lösningsarkitekt                              169
Kravanalytiker                                148
Solution Architect                            100
Data Scientist                                 79
IT-arkitekt                                    74
Jönköping: Senior IT-arkitekt hybridtjänst     72
IT-konsult                                     66
Sales Representative                           65
Business Analyst                               61
Enterprise Architect                           52
Lösningsarkitekt till Softhouse i Luleå        49
Senior Kravanalytiker                          44
Senior Data Scientist                          44
Informationsarkitekt                           43
Application Consultant                         38
Enterprisearkitekt                             37
Configuration Manager                          34
Data Analyst       

In [49]:
# Yrkesgrupper enligt SSYK (occupation)
unique_occupations = df["occupation"].nunique()

print(f"Antal unika yrkesgrupper: {unique_occupations}")
print("---------------------------")

print(f"Genomsnittligt antal annonser per yrke: {len(df) / unique_occupations:.2f}")
print("---------------------------")

print(df["occupation"].value_counts().head(20))
print("---------------------------")

Antal unika yrken: 16
---------------------------
Genomsnittligt antal annonser per yrke: 709.06
---------------------------
occupation
IT-arkitekt/Lösningsarkitekt       3400
Affärskonsult, IT                  1582
Systemanalytiker/Systemutredare    1411
Kravanalytiker                     1067
Systemarkitekt                      787
Verksamhetskonsult, IT              636
Enterprise Architect                528
IT-arkitekt                         506
Infrastrukturarkitekt               481
Data scientist                      441
Configuration Manager/CM            199
IT-utredare                         152
Metodanalytiker, IT                 110
Informationsarkitekt                 28
Informatiker                         11
Nätverksarkitekt                      6
Name: count, dtype: int64
---------------------------


In [50]:
# Geografisk fördelning
# enligt kommuner
print("Antal unika kommuner:", df["municipality"].nunique())
print("---------------------------")
print(df["municipality"].value_counts().head(20))
print("---------------------------")

# enligt region
print("Antal unika regioner:", df["region"].nunique())
print("---------------------------")
print(df["region"].value_counts().head(20))
print("---------------------------")

Antal unika kommuner: 120
---------------------------
municipality
Stockholm      3852
Göteborg       1190
Malmö           514
Solna           454
Linköping       292
Sundbyberg      285
Jönköping       273
Luleå           265
Lund            231
Södertälje      187
Norrköping      165
Örebro          140
Sundsvall       128
Västerås        124
Helsingborg     102
Växjö           100
Uppsala          93
Enköping         90
Karlstad         79
Umeå             67
Name: count, dtype: int64
---------------------------
Antal unika regioner: 21
---------------------------
region
Stockholms län          4922
Västra Götalands län    1468
Skåne län                884
Östergötlands län        541
Jönköpings län           301
Norrbottens län          290
Uppsala län              195
Örebro län               175
Västernorrlands län      162
Kronobergs län           159
Västmanlands län         159
Västerbottens län         99
Värmlands län             80
Dalarnas län              78
Kalmar län   

In [51]:
# datumfördelning
df["published"] = pd.to_datetime(df["published"])

df["year"] = df["published"].dt.year
df["month"] = df["published"].dt.month

# Annonser per år
print(df["year"].value_counts().sort_index())
print("---------------------------")

# Annonser per månad
print(df["month"].value_counts().sort_index())
print("---------------------------")

# Annonser per år + månad
df["year_month"] = df["published"].dt.to_period("M")
monthly_ads = df["year_month"].value_counts().sort_index()

print(monthly_ads)
print("---------------------------")

# Vilken månad + år hade mest annonser
print(monthly_ads.idxmax())
print(monthly_ads.max())
print("---------------------------")


year
2025    6938
2026    4407
Name: count, dtype: int64
---------------------------
month
1     1067
2     1155
3     1033
4     1515
5     1466
6     1079
7      664
8      620
9      889
10     667
11     477
12     713
Name: count, dtype: int64
---------------------------
year_month
2025-01    482
2025-02    474
2025-03    425
2025-04    743
2025-05    862
2025-06    556
2025-07    462
2025-08    357
2025-09    720
2025-10    667
2025-11    477
2025-12    713
2026-01    585
2026-02    681
2026-03    608
2026-04    772
2026-05    604
2026-06    523
2026-07    202
2026-08    263
2026-09    169
Freq: M, Name: count, dtype: int64
---------------------------
2025-05
862
---------------------------


In [52]:
#Tabell med annonser per månad per år
year_month_table = pd.crosstab(
    df["month"],
    df["year"]
)

print(year_month_table)

year   2025  2026
month            
1       482   585
2       474   681
3       425   608
4       743   772
5       862   604
6       556   523
7       462   202
8       357   263
9       720   169
10      667     0
11      477     0
12      713     0


In [53]:
# datumintervallet
print("Första annonsen:", df["published"].min())
print("Senaste annonsen:", df["published"].max())

Första annonsen: 2025-01-02 08:28:27
Senaste annonsen: 2026-09-28 16:06:38


In [54]:
# beskrivningarna
# längd
df["description_length"] = df["description"].str.len()
print(df["description_length"].describe())
print("---------------------------")

# antal tecken
df["word_count"] = (
    df["description"]
    .str.split()
    .str.len()
)
print(df["word_count"].describe())
print("---------------------------")

# tomma descriptions
print(
    "Tomma descriptions:",
    df["description"].isna().sum()
)
print(
    "Descriptions < 100 tecken:",
    (df["description_length"] < 100).sum()
)

count    11345.000000
mean      3172.007933
std       1455.751675
min          0.000000
25%       2132.000000
50%       3112.000000
75%       4241.000000
max       6506.000000
Name: description_length, dtype: float64
---------------------------
count    11345.000000
mean       451.935037
std        206.555819
min          0.000000
25%        301.000000
50%        445.000000
75%        609.000000
max        978.000000
Name: word_count, dtype: float64
---------------------------
Tomma descriptions: 0
Descriptions < 100 tecken: 20


## Taxonomi EDA

### Taxonomi grundläggande struktur

In [55]:
# Snabb överblick
print(f"Taxonomi innehåller {tax.shape[0]} rader (annonser) och {tax.shape[1]} kolumner.\n")
print("---------------------------")
print(tax.info())
print("---------------------------")

print("\n Taxonmi kolumner utdrag:")
print(tax.head())
print("---------------------------")

print("\nFeatures i taxonomi:\n", tax.columns)
print("---------------------------")

Taxonomi innehåller 204 rader (annonser) och 3 kolumner.

---------------------------
<class 'pandas.DataFrame'>
RangeIndex: 204 entries, 0 to 203
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   skill_id   204 non-null    str   
 1   skill      204 non-null    str   
 2   headlines  204 non-null    object
dtypes: object(1), str(2)
memory usage: 4.9+ KB
None
---------------------------

 Taxonmi kolumner utdrag:
       skill_id                                          skill  \
0  13ad_dCC_Tn6               OOS, styr- och utvecklingsmodell   
1  16ZN_3Zp_bYV                            Odoo, molnplattform   
2  19PK_uxU_Wzy                          Bower, pakethanterare   
3  1CmG_8hR_igc                               Bluetooth-teknik   
4  1QYQ_Xka_TQ5  Microsoft Autopilot, systemutvecklingsverktyg   

                                        headlines  
0                 [Styr- och utvecklingsmodeller]  
1                

### Datakvalitet

In [56]:
# Saknade värden
missing = tax.isna().sum()

print(missing)
print("---------------------------")

missing_percentage = (
    tax.isna().mean() * 100
).sort_values(ascending=False)

print(missing_percentage)
print("---------------------------")

#Dubletter
print("Unika skill-ID:", tax["skill_id"].nunique())
print("Totalt antal annonser:", len(tax))

skill_id     0
skill        0
headlines    0
dtype: int64
---------------------------
skill_id     0.0
skill        0.0
headlines    0.0
dtype: float64
---------------------------
Unika skill-ID: 204
Totalt antal annonser: 204


In [61]:
skill_headline = tax[["skill", "headlines"]].copy()

print(skill_headline.to_string(index=False))

                                                             skill                                      headlines
                                  OOS, styr- och utvecklingsmodell                [Styr- och utvecklingsmodeller]
                                               Odoo, molnplattform                      [Applikationsplattformar]
                                             Bower, pakethanterare [Programmerings- och systemutvecklingsverktyg]
                                                  Bluetooth-teknik                      [Kommunikationsprotokoll]
                     Microsoft Autopilot, systemutvecklingsverktyg [Programmerings- och systemutvecklingsverktyg]
            Office SharePoint Server (MOSS), applikationsplattform                      [Applikationsplattformar]
                                  Ledningssystem (ISO 9000-serien)                              [Kvalitetssystem]
                                                        WAP-teknik                      